In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

In [3]:
dataset = pd.read_csv('heart_disease.csv')
print(dataset.head())

    Age  Gender  Blood Pressure  Cholesterol Level Exercise Habits Smoking  \
0  56.0    Male           153.0              155.0            High     Yes   
1  69.0  Female           146.0              286.0            High      No   
2  46.0    Male           126.0              216.0             Low      No   
3  32.0  Female           122.0              293.0            High     Yes   
4  60.0    Male           166.0              242.0             Low     Yes   

  Family Heart Disease Diabetes        BMI High Blood Pressure  ...  \
0                  Yes       No  24.991591                 Yes  ...   
1                  Yes      Yes  25.221799                  No  ...   
2                   No       No  29.855447                  No  ...   
3                  Yes       No  24.130477                 Yes  ...   
4                  Yes      Yes  20.486289                 Yes  ...   

  High LDL Cholesterol Alcohol Consumption Stress Level Sleep Hours  \
0                   No           

In [4]:
print("\nDataset shape:", dataset.shape)
print("\nDataset info:")
print(dataset.info())
print("\nMissing values:")
print(dataset.isnull().sum())


Dataset shape: (10000, 21)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   9971 non-null   float64
 1   Gender                9981 non-null   object 
 2   Blood Pressure        9981 non-null   float64
 3   Cholesterol Level     9970 non-null   float64
 4   Exercise Habits       9975 non-null   object 
 5   Smoking               9975 non-null   object 
 6   Family Heart Disease  9979 non-null   object 
 7   Diabetes              9970 non-null   object 
 8   BMI                   9978 non-null   float64
 9   High Blood Pressure   9974 non-null   object 
 10  Low HDL Cholesterol   9975 non-null   object 
 11  High LDL Cholesterol  9974 non-null   object 
 12  Alcohol Consumption   7414 non-null   object 
 13  Stress Level          9978 non-null   object 
 14  Sleep Hours           9975 n

In [5]:
# Check class imbalance in target variable
print("\n=== CLASS IMBALANCE ANALYSIS ===")
target_column = dataset.columns[-1]  # Assuming last column is target
print(f"Target variable: {target_column}")
print("\nClass distribution:")
class_dist = dataset[target_column].value_counts()
print(class_dist)
print("\nClass distribution (%):")
print((class_dist / len(dataset) * 100).round(2))
print("\n⚠️ WARNING: Severe class imbalance detected!")
print("A model predicting all 'No' would get 80% accuracy but be useless.")
print("Solution: Using class_weight='balanced' to penalize misclassification of minority class.")


=== CLASS IMBALANCE ANALYSIS ===
Target variable: Heart Disease Status

Class distribution:
Heart Disease Status
No     8000
Yes    2000
Name: count, dtype: int64

Class distribution (%):
Heart Disease Status
No     80.0
Yes    20.0
Name: count, dtype: float64

⚠️ WARNING: Severe class imbalance detected!
A model predicting all 'No' would get 80% accuracy but be useless.
Solution: Using class_weight='balanced' to penalize misclassification of minority class.


In [6]:
# Check for and drop duplicates
print(f"Initial dataset shape: {dataset.shape}")
print(f"Duplicate rows: {dataset.duplicated().sum()}")

dataset = dataset.drop_duplicates()
print(f"Dataset shape after dropping duplicates: {dataset.shape}")
print(f"Rows removed: {dataset.shape[0] - dataset.shape[0]}")
dataset = dataset.reset_index(drop=True)

Initial dataset shape: (10000, 21)
Duplicate rows: 0
Dataset shape after dropping duplicates: (10000, 21)
Rows removed: 0


In [ ]:
# Handle Missing Values
print("\n=== HANDLING MISSING VALUES ===\n")

# Show missing values summary
print("Missing values before imputation:")
missing_vals = dataset.isnull().sum()
missing_pct = (missing_vals / len(dataset) * 100).round(2)
missing_df = pd.DataFrame({'Column': missing_vals.index, 'Missing_Count': missing_vals.values, 'Missing_%': missing_pct.values})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df.to_string(index=False))

# Identify numerical and categorical columns
numerical_cols = dataset.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = dataset.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")

# Special handling for Alcohol Consumption (if exists)
alcohol_col = 'Alcohol Consumption' if 'Alcohol Consumption' in dataset.columns else None

if alcohol_col:
    print(f"\n📌 Special handling for '{alcohol_col}':")
    print(f"   Missing: {dataset[alcohol_col].isnull().sum()} ({(dataset[alcohol_col].isnull().sum()/len(dataset)*100):.1f}%)")
    print(f"   → Filling with 'Unknown' category (not guessing)")
    dataset[alcohol_col] = dataset[alcohol_col].fillna('Unknown')

# Fill missing values in other columns
for col in numerical_cols:
    if dataset[col].isnull().sum() > 0 and col != target_column:
        median_val = dataset[col].median()
        dataset[col] = dataset[col].fillna(median_val)
        print(f"   Filled {col} (numerical) with median: {median_val:.2f}")

for col in categorical_cols:
    if dataset[col].isnull().sum() > 0 and col != target_column:
        mode_val = dataset[col].mode()[0]
        dataset[col] = dataset[col].fillna(mode_val)
        print(f"   Filled {col} (categorical) with mode: {mode_val}")

# Verify no missing values remain
print(f"\n✅ Missing values after imputation: {dataset.isnull().sum().sum()}")
print("Data cleaning complete!")